# โครงการวิจัย: การพัฒนาแพลตฟอร์มปัญญาประดิษฐ์บนอุปกรณ์พกพาสำหรับจำแนกแมลงศัตรูข้าวและให้คำแนะนำเชิงปฏิบัติแบบเรียลไทม์

สมุดบันทึกนี้ออกแบบมาเพื่อการฝึกสอน ตรวจสอบ และแปลงโมเดลปัญญาประดิษฐ์สำหรับการจำแนกแมลงศัตรูข้าว 22 ชนิดของประเทศไทย เพื่อรองรับการนำไปติดตั้งบนแอปพลิเคชันมือถือ (TensorFlow Lite)

### 📋 สถาปัตยกรรมและเวิร์กโฟลว์ของสมุดบันทึก
1. **ระบบเตรียมความพร้อม**: เชื่อมต่อ Google Drive และตั้งค่าสภาพแวดล้อม (TensorFlow T4 GPU)
2. **พาธสำหรับ Dataset**: ตรวจสอบตำแหน่งโฟลเดอร์ภาพและไฟล์คำแนะนำ
3. **ตรวจสอบจำนวนภาพสองแหล่งข้อมูลใน GBIF API**: ดึงชื่อวิทยาศาสตร์ทั้ง 2 แหล่งข้อมูล (แบบดั้งเดิม vs แบบใหม่) และวิเคราะห์จำนวนรูปภาพถ่ายที่มีอยู่บนฐานข้อมูลสากล GBIF
4. **การทำความสะอาดภาพ**: ตรวจสอบภาพชำรุดและลบไฟล์ภาพที่ไม่สมบูรณ์
5. **การพรีโพรเซสภาพ**: ปรับขนาดภาพด้วยแนวทาง **Letterbox Resizing with Padding** ขนาด 224x224 พิกเซล เพื่อรักษาสันฐานวิทยาแมลง
6. **ท่อส่งข้อมูลการเทรน**: ทำการแบ่งชุดข้อมูล (Train 80% / Val 20%) และเพิ่มความทนทานโมเดลด้วย **Data Augmentation**
7. **การฝึกสอนและเปรียบเทียบโมเดล (Two-Phase Transfer Learning)**:
   - **โมเดลที่เปรียบเทียบ**: `MobileNetV2`, `EfficientNetB0`, `ResNet50V2`
   - **เฟสที่ 1 (Feature Extraction)**: ตรึงน้ำหนักของ Base Network และฝึกสอนเฉพาะชั้นจำแนก
   - **เฟสที่ 2 (Fine-Tuning)**: ปลดล็อกเลเยอร์ระดับสูง (Top Layers) ของแบบจำลองและจูนรายละเอียดด้วยอัตราการเรียนรู้ระดับต่ำสุด
8. **การประเมินและเปรียบเทียบ**: พล็อตและวิเคราะห์กราฟความถูกต้อง (Accuracy) และความสูญเสีย (Loss)
9. **การแปลงน้ำหนักโมเดล**: ส่งออกโมเดลที่ดีที่สุดเป็นไฟล์ `.tflite` สำหรับติดตั้งบนอุปกรณ์พกพา
10. **ระบบผู้เชี่ยวชาญ (Inference with Real-time Advice)**: การจำแนกภาพแมลงเดี่ยวและเชื่อมโยงคำแนะนำเกษตรจากไฟล์ `recommendations.json` ทันที

## 🛠️ ส่วนที่ 1: การโหลดไลบรารีที่จำเป็นและการตั้งค่าอุปกรณ์ประมวลผล

In [1]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import urllib.request
import urllib.parse
from PIL import Image
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print("TensorFlow Version:", tf.__version__)
device_name = tf.test.gpu_device_name()
if device_name != '/device:GPU:0':
    print('WARNING: GPU device not found. Please change runtime to GPU (T4).')
else:
    print('Success: Found GPU at:', device_name)

TensorFlow Version: 2.20.0


## 📂 ส่วนที่ 2: เชื่อมต่อ Google Drive และตั้งค่าพาธสำหรับ Dataset

In [2]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # พาธมาตรฐานเมื่อนำโฟลเดอร์ไปวางใน Google Drive
    DATASET_DIR = '/content/drive/MyDrive/Colab Notebooks/insect/dataset/images'
    RECOMMENDATIONS_PATH = '/content/drive/MyDrive/Colab Notebooks/insect/data/recommendations.json'
except:
    print("Running on local machine or standard workspace.")
    DATASET_DIR = 'dataset/images'
    RECOMMENDATIONS_PATH = 'data/recommendations.json'

print("Checking Dataset location:", DATASET_DIR)
if os.path.exists(DATASET_DIR):
    classes = sorted(os.listdir(DATASET_DIR))
    print(f"Found {len(classes)} classes of rice insect pests:")
    for idx, cls in enumerate(classes):
        count = len(os.listdir(os.path.join(DATASET_DIR, cls)))
        print(f"{idx+1}. {cls}: {count} images")
else:
    print("Error: Dataset directory not found. Please upload dataset/images to your workspace or Google Drive.")

Mounted at /content/drive
Checking Dataset location: /content/drive/MyDrive/Colab Notebooks/insect/dataset/images
Error: Dataset directory not found. Please upload dataset/images to your workspace or Google Drive.


## 🌐 ส่วนที่ 3: การตรวจสอบและแปลงข้อมูลจำนวนภาพบนคลังข้อมูลโลก GBIF (Query GBIF Counts with TaxonKey Match)
ส่วนนี้จะทำการอ่านไฟล์คำแนะนำภาษาไทยและดึงชื่อวิทยาศาสตร์ใหม่ (scientific_name_2) จากนั้นทำการค้นหาข้อมูลทางอนุกรมวิธานผ่าน **GBIF Species Match API** เพื่อแปลงชื่อให้เป็นรหัสอ้างอิง `taxonKey` สากล (ช่วยแก้ไขปัญหาการสะกดผิดหรือมีชื่อผู้ค้นพบปะปน) และส่งสืบค้นไปยัง GBIF Occurrence API เพื่อสรุปจำนวนภาพและข้อมูลอนุกรมวิธานในรูปแบบไฟล์ JSON (`gbif_summary.json`)


In [4]:
print("Reading recommendations.json from:", RECOMMENDATIONS_PATH)
if os.path.exists(RECOMMENDATIONS_PATH):
    with open(RECOMMENDATIONS_PATH, 'r', encoding='utf-8') as f:
        pest_db = json.load(f)
    
    gbif_results = []
    gbif_summary = {}
    print("Querying GBIF Backbone Taxonomy and Occurrence APIs (Option 1 - TaxonKey match)...\n")
    
    for thai_name, info in pest_db.items():
        name2 = info.get('scientific_name_2', '')
        
        taxon_key = None
        matched_sci_name = ""
        status = ""
        rank = ""
        confidence = 0
        match_type = ""
        kingdom = ""
        phylum = ""
        class_name = ""
        order = ""
        family = ""
        genus = ""
        
        if name2:
            match_url = f"https://api.gbif.org/v1/species/match?name={urllib.parse.quote(name2)}"
            try:
                with urllib.request.urlopen(match_url, timeout=10) as r:
                    res = json.loads(r.read().decode('utf-8'))
                    taxon_key = res.get('usageKey') or res.get('speciesKey')
                    matched_sci_name = res.get('scientificName', '')
                    status = res.get('status', '')
                    rank = res.get('rank', '')
                    confidence = res.get('confidence', 0)
                    match_type = res.get('matchType', '')
                    kingdom = res.get('kingdom', '')
                    phylum = res.get('phylum', '')
                    class_name = res.get('class', '')
                    order = res.get('order', '')
                    family = res.get('family', '')
                    genus = res.get('genus', '')
            except Exception as e:
                print(f"Error matching {name2}: {e}")
                
        # คิวรีหาจำนวนภาพโดยใช้ taxonKey
        count = 0
        if taxon_key:
            occ_url = f"https://api.gbif.org/v1/occurrence/search?taxonKey={taxon_key}&mediaType=StillImage&limit=0"
            try:
                with urllib.request.urlopen(occ_url, timeout=10) as r:
                    count = json.loads(r.read().decode('utf-8')).get('count', 0)
            except Exception as e:
                count = "Error"
        else:
            count = "-"
            
        gbif_results.append({
            "ชื่อภาษาไทย": thai_name,
            "ชื่อวิทยาศาสตร์ 2 (ใหม่)": name2,
            "ชื่อวิทย์ที่ตรงบน GBIF": matched_sci_name,
            "Taxon Key": taxon_key,
            "จำนวนภาพ (GBIF)": count
        })
        print(f"- {thai_name}: {name2} -> GBIF: {matched_sci_name} ({count} ภาพ)")
        
        # เก็บข้อมูลสรุปลง dict เพื่อแปลงเป็น JSON
        gbif_summary[thai_name] = {
            "thai_name": thai_name,
            "image_count": count,
            "scientific_name_2": name2,
            "gbif_scientific_name": matched_sci_name,
            "taxon_key": taxon_key,
            "status": status,
            "rank": rank,
            "confidence": confidence,
            "match_type": match_type,
            "kingdom": kingdom,
            "phylum": phylum,
            "class": class_name,
            "order": order,
            "family": family,
            "genus": genus
        }
        
    # แปลงเป็น DataFrame และแสดงตาราง
    df_gbif = pd.DataFrame(gbif_results)
    print("\n=== ตารางสรุปจำนวนภาพถ่ายจากการใช้ TaxonKey แมชชิ่งบน GBIF ===")
    display(df_gbif)
    
    # บันทึกไฟล์ JSON สรุปข้อมูลแมลงใหม่ตามคำร้องขอ
    summary_json_path = os.path.join(os.path.dirname(RECOMMENDATIONS_PATH), "gbif_summary.json")
    with open(summary_json_path, 'w', encoding='utf-8') as f:
        json.dump(gbif_summary, f, indent=4, ensure_ascii=False)
    print(f"\nบันทึกข้อมูลสรุปในรูปแบบ JSON สำเร็จที่: {summary_json_path}")
else:
    print(f"Error: recommendations.json not found at {RECOMMENDATIONS_PATH}")

Reading recommendations.json from: /content/drive/MyDrive/Colab Notebooks/insect/data/recommendations.json
Querying GBIF Backbone Taxonomy and Occurrence APIs (Option 1 - TaxonKey match)...

- ด้วงงวงกินรากข้าว: Hydronomidius molitor Faust -> GBIF: Hydronomidius molitor Faust, 1898 (0 ภาพ)
- ด้วงดำ: Heteronychus lioderes Redtenbacher -> GBIF: Heteronychus lioderes Redtenbacher, 1868 (0 ภาพ)
- มวนง่าม: Tetroda denticulifera (Berg) -> GBIF: Tetroda Amyot & Serville, 1843 (46 ภาพ)
- หนอนกระทู้กล้า: Spodoptera mauritia (Boisduval) -> GBIF: Spodoptera mauritia (Boisduval, 1833) (2488 ภาพ)
- หนอนกระทู้คอรวง: Mythimna separata (Walker) -> GBIF: Mythimna separata (Walker, 1865) (860 ภาพ)
- หนอนกอข้าวสีครีม: Scirpophaga incertulas (Walker) -> GBIF: Scirpophaga incertulas Walker, 1863 (438 ภาพ)
- หนอนกอสีชมพู: Sesamia inferens (Walker) -> GBIF: Sesamia inferens (Walker, 1856) (96 ภาพ)
- หนอนกอแถบลายสีม่วง: Chilo polychrysus (Meyrick) -> GBIF: Chilo polychrysa Meyrick, 1932 (0 ภาพ)
- หนอนกอแถบลาย

,ชื่อภาษาไทย,ชื่อวิทยาศาสตร์ 2 (ใหม่),ชื่อวิทย์ที่ตรงบน GBIF,Taxon Key,จำนวนภาพ (GBIF)
0,ด้วงงวงกินรากข้าว,Hydronomidius molitor Faust,"Hydronomidius molitor Faust, 1898",1247989,0
1,ด้วงดำ,Heteronychus lioderes Redtenbacher,"Heteronychus lioderes Redtenbacher, 1868",4995399,0
2,มวนง่าม,Tetroda denticulifera (Berg),"Tetroda Amyot & Serville, 1843",4783006,46
3,หนอนกระทู้กล้า,Spodoptera mauritia (Boisduval),"Spodoptera mauritia (Boisduval, 1833)",5109848,2488
4,หนอนกระทู้คอรวง,Mythimna separata (Walker),"Mythimna separata (Walker, 1865)",5802396,860
5,หนอนกอข้าวสีครีม,Scirpophaga incertulas (Walker),"Scirpophaga incertulas Walker, 1863",1881293,438
6,หนอนกอสีชมพู,Sesamia inferens (Walker),"Sesamia inferens (Walker, 1856)",1762353,96
7,หนอนกอแถบลายสีม่วง,Chilo polychrysus (Meyrick),"Chilo polychrysa Meyrick, 1932",1883232,0
8,หนอนกอแถบลายเล็ก,Chilo suppressalis (Walker),"Chilo suppressalis Walker, 1863",1883226,56
9,หนอนปลอกข้าว,Nymphula depunctalis Guenée,"Nymphula Schrank, 1802",1884090,7081



บันทึกข้อมูลสรุปในรูปแบบ JSON สำเร็จที่: /content/drive/MyDrive/Colab Notebooks/insect/data/gbif_summary.json


## 🧹 ส่วนที่ 4: การดาวน์โหลดภาพจาก GBIF และตรวจสอบความถูกต้องของข้อมูล (Image Download & Data Integrity Check)
ส่วนนี้จะทำการดึงข้อมูลสายพันธุ์จากไฟล์ `gbif_summary.json` และส่งคำขอสืบค้นไปยัง GBIF API เพื่อดาวน์โหลดภาพแมลงศัตรูข้าวแต่ละชนิดสูงสุด 100 ภาพต่อชนิด โดยบันทึกไฟล์ด้วยชื่อรหัส `key` ของ GBIF (เพื่อป้องกันการดาวน์โหลดซ้ำ) พร้อมกับตรวจสอบการชำรุดของภาพและทำการปรับขนาดภาพแบบรักษาอัตราส่วน (Letterbox Resizing with Padding ขนาด 224x224 พิกเซล) ก่อนจัดเก็บลงดิสก์

In [6]:
import os
import json
import urllib.request
import urllib.parse
import io
import numpy as np
import cv2
from PIL import Image

def download_and_resize_gbif_images(summary_json_path, dataset_dir, limit_per_species=150, target_size=(224, 224), padding_color=(255, 255, 255)):
    if not os.path.exists(summary_json_path):
        print(f"Error: {summary_json_path} not found.")
        return
        
    with open(summary_json_path, 'r', encoding='utf-8') as f:
        summary_db = json.load(f)
        
    print(f"Loaded {len(summary_db)} species from summary JSON.")
    
    # วนลูปดาวน์โหลดภาพแต่ละสปีชีส์
    for thai_name, info in summary_db.items():
        taxon_key = info.get('taxon_key')
        image_count = info.get('image_count', 0)
        
        if not taxon_key or image_count == 0 or image_count == '-':
            print(f"\n- {thai_name}: ไม่มีภาพบน GBIF (ข้าม)")
            continue
            
        species_dir = os.path.join(dataset_dir, thai_name)
        os.makedirs(species_dir, exist_ok=True)
        
        # สืบค้นรายการ Occurrence สูงสุด 150 รายการ
        occ_limit = min(limit_per_species, int(image_count))
        print(f"\n- {thai_name}: กำลังดึงรายการภาพจาก GBIF (ต้องการสูงสุด {occ_limit} ภาพ)...")
        
        occ_url = f"https://api.gbif.org/v1/occurrence/search?taxonKey={taxon_key}&mediaType=StillImage&limit={limit_per_species}"
        try:
            with urllib.request.urlopen(occ_url, timeout=10) as r:
                res = json.loads(r.read().decode('utf-8'))
                occurrences = res.get('results', [])
        except Exception as e:
            print(f"  Error fetching occurrences for {thai_name}: {e}")
            continue
            
        downloaded_count = 0
        skipped_count = 0
        failed_count = 0
        
        for occ in occurrences:
            # ตรวจสอบขีดจำกัด
            if downloaded_count >= limit_per_species:
                break
                
            occ_key = occ.get('key')
            media_list = occ.get('media', [])
            if not occ_key or not media_list:
                continue
                
            # หา StillImage URL แรก
            img_url = None
            for m in media_list:
                if m.get('type') == 'StillImage' and m.get('identifier'):
                    img_url = m.get('identifier')
                    break
                    
            if not img_url:
                continue
                
            save_path = os.path.join(species_dir, f"{occ_key}.jpg")
            
            # ตรวจสอบการดาวน์โหลดซ้ำ
            if os.path.exists(save_path):
                # ตรวจสอบว่าไฟล์ที่โหลดมาก่อนหน้านี้ชำรุดหรือไม่
                try:
                    with Image.open(save_path) as img:
                        img.verify()
                    skipped_count += 1
                    downloaded_count += 1
                    continue
                except:
                    print(f"  ตรวจพบภาพเดิมชำรุด จะทำการดาวน์โหลดใหม่: {save_path}")
                    try:
                        os.remove(save_path)
                    except:
                        pass
                        
            # ดาวน์โหลดและตรวจสภาพภาพชำรุด
            try:
                # สร้าง Request พร้อม User-Agent เพื่อหลีกเลี่ยงการโดนบล็อก
                req = urllib.request.Request(
                    img_url, 
                    headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
                )
                with urllib.request.urlopen(req, timeout=15) as resp:
                    img_bytes = resp.read()
                    
                # ตรวจสอบความชำรุดของภาพด้วย PIL
                img_pil = Image.open(io.BytesIO(img_bytes))
                img_pil.verify()
                
                # ถอดรหัสภาพด้วย OpenCV
                img_bgr = cv2.imdecode(np.frombuffer(img_bytes, np.uint8), cv2.IMREAD_COLOR)
                if img_bgr is None:
                    raise ValueError("OpenCV imdecode returned None")
                    
                # ปรับขนาดภาพแบบ Letterbox Resizing with Padding (รักษารูปทรงสัณฐาน)
                h, w = img_bgr.shape[:2]
                target_w, target_h = target_size
                scale = min(target_w / w, target_h / h)
                new_w = int(w * scale)
                new_h = int(h * scale)
                
                resized = cv2.resize(img_bgr, (new_w, new_h), interpolation=cv2.INTER_AREA)
                padded = np.full((target_h, target_w, 3), padding_color, dtype=np.uint8)
                
                x_offset = (target_w - new_w) // 2
                y_offset = (target_h - new_h) // 2
                padded[y_offset:y_offset+new_h, x_offset:x_offset+new_w] = resized
                
                # บันทึกภาพลงดิสก์
                cv2.imwrite(save_path, padded)
                downloaded_count += 1
            except Exception as e:
                failed_count += 1
                
        print(f"  -> สรุป: โหลดสำเร็จ {downloaded_count} ภาพ (ข้ามภาพซ้ำ {skipped_count} ภาพ, ล้มเหลว {failed_count} ภาพ)")

def check_and_clean_images(directory):
    corrupted_files = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.webp', '.bmp')):
                file_path = os.path.join(root, file)
                try:
                    with Image.open(file_path) as img:
                        img.verify()
                except Exception as e:
                    print(f"Corrupted image detected: {file_path} - Error: {e}")
                    corrupted_files.append(file_path)
    return corrupted_files

# กำหนดโฟลเดอร์เก็บข้อมูลภาพ
if 'DATASET_DIR' not in globals():
    DATASET_DIR = 'dataset/images'
if 'RECOMMENDATIONS_PATH' not in globals():
    RECOMMENDATIONS_PATH = 'data/recommendations.json'

summary_json = os.path.join(os.path.dirname(RECOMMENDATIONS_PATH), "gbif_summary.json")

print("Starting GBIF images download and validation process...")
download_and_resize_gbif_images(summary_json, DATASET_DIR, limit_per_species=150)

print("\nScanning dataset for corrupted images...")
corrupted = check_and_clean_images(DATASET_DIR)
if len(corrupted) == 0:
    print("All images verified successfully. No corrupted files found!")
else:
    print(f"Found {len(corrupted)} corrupted files.")
    # ลบภาพชำรุดออกเพื่อความสะอาดของข้อมูล
    for path in corrupted:
        try:
            os.remove(path)
            print(f"Deleted corrupted file: {path}")
        except Exception as e:
            print(f"Failed to delete {path}: {e}")

Starting GBIF images download and validation process...
Loaded 22 species from summary JSON.

- ด้วงงวงกินรากข้าว: ไม่มีภาพบน GBIF (ข้าม)

- ด้วงดำ: ไม่มีภาพบน GBIF (ข้าม)

- มวนง่าม: กำลังดึงรายการภาพจาก GBIF (ต้องการสูงสุด 46 ภาพ)...
  -> สรุป: โหลดสำเร็จ 46 ภาพ (ข้ามภาพซ้ำ 46 ภาพ, ล้มเหลว 0 ภาพ)

- หนอนกระทู้กล้า: กำลังดึงรายการภาพจาก GBIF (ต้องการสูงสุด 150 ภาพ)...
  -> สรุป: โหลดสำเร็จ 150 ภาพ (ข้ามภาพซ้ำ 100 ภาพ, ล้มเหลว 0 ภาพ)

- หนอนกระทู้คอรวง: กำลังดึงรายการภาพจาก GBIF (ต้องการสูงสุด 150 ภาพ)...
  -> สรุป: โหลดสำเร็จ 148 ภาพ (ข้ามภาพซ้ำ 98 ภาพ, ล้มเหลว 2 ภาพ)

- หนอนกอข้าวสีครีม: กำลังดึงรายการภาพจาก GBIF (ต้องการสูงสุด 150 ภาพ)...
  -> สรุป: โหลดสำเร็จ 148 ภาพ (ข้ามภาพซ้ำ 100 ภาพ, ล้มเหลว 2 ภาพ)

- หนอนกอสีชมพู: กำลังดึงรายการภาพจาก GBIF (ต้องการสูงสุด 96 ภาพ)...
  -> สรุป: โหลดสำเร็จ 65 ภาพ (ข้ามภาพซ้ำ 65 ภาพ, ล้มเหลว 31 ภาพ)

- หนอนกอแถบลายสีม่วง: ไม่มีภาพบน GBIF (ข้าม)

- หนอนกอแถบลายเล็ก: กำลังดึงรายการภาพจาก GBIF (ต้องการสูงสุด 56 ภาพ)...
  -> สรุป: โหลดสำเร็จ 56 ภาพ (ข

## 📐 ส่วนที่ 5: การประมวลผลขนาดภาพโดยรักษาอัตราส่วน (Letterbox Resizing with Padding)
การย่อภาพแบบปกติอาจทำให้ลักษณะทางสัณฐานวิทยาของแมลง เช่น ความกว้างลำตัว ขา หรือหนวดเกิดการบิดเบี้ยวได้ เราจึงใช้วิธี **Letterbox** โดยการปรับให้ความกว้างหรือความสูงเท่ากับขนาดเป้าหมายก่อนแล้วเติมขอบว่างส่วนที่ขาดด้วยสีขาว/ดำ

In [ ]:
def letterbox_image(image_path, target_size=(224, 224), padding_color=(255, 255, 255)):
    """
    ฟังก์ชันปรับขนาดภาพโดยรักษาอัตราส่วนและเติมขอบว่าง (Letterbox with Padding)
    """
    img = cv2.imread(image_path)
    if img is None:
        return None
    
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    target_w, target_h = target_size

    # คำนวณสัดส่วนสเกลที่ต้องปรับ
    scale = min(target_w / w, target_h / h)
    new_w = int(w * scale)
    new_h = int(h * scale)

    # ปรับขนาดภาพแบบรักษาอัตราส่วน
    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)

    # สร้างกรอบภาพขนาดเป้าหมายพร้อมเติมพื้นหลังสีที่กำหนด
    padded = np.full((target_h, target_w, 3), padding_color, dtype=np.uint8)

    # คำนวณหาพิกัดกึ่งกลางเพื่อเอาภาพสเกลวางตรงกลาง
    x_offset = (target_w - new_w) // 2
    y_offset = (target_h - new_h) // 2

    # วางภาพทับลงในกรอบพิกัดกึ่งกลาง
    padded[y_offset:y_offset+new_h, x_offset:x_offset+new_w] = resized
    
    return padded

# แสดงภาพตัวอย่างการทำ Letterbox
if os.path.exists(DATASET_DIR):
    first_class = classes[0]
    class_dir = os.path.join(DATASET_DIR, first_class)
    first_image = os.path.join(class_dir, os.listdir(class_dir)[0])
    
    original_img = Image.open(first_image)
    letterboxed_img = letterbox_image(first_image, target_size=(224, 224))
    
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.title(f"Original ({original_img.size[0]}x{original_img.size[1]})")
    plt.imshow(original_img)
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.title("Letterboxed with Padding (224x224)")
    plt.imshow(letterboxed_img)
    plt.axis('off')
    plt.show()

## 🔄 ส่วนที่ 6: การจัดแบ่งชุดข้อมูลการเทรนและ Data Augmentation
เราจะประยุกต์ใช้ ImageDataGenerator ร่วมกับการหมุนสุ่ม พลิกภาพ และการสเกลขนาดสำหรับ Train และ Validation ในอัตราส่วน 80:20 เพื่อลดโอกาสเกิด Overfitting

In [ ]:
BATCH_SIZE = 16
IMAGE_SIZE = (224, 224)

# ใช้เทคนิคการขยายข้อมูล (Data Augmentation) สำหรับชุดฝึกสอน
train_datagen = ImageDataGenerator(
    rescale=1./255,               # ทำการ Normalization ปรับช่วงสีจาก 0-255 เป็น 0-1
    rotation_range=20,            # สุ่มหมุนภาพไม่เกิน 20 องศา
    width_shift_range=0.1,        # สุ่มเลื่อนด้านข้าง
    height_shift_range=0.1,       # สุ่มเลื่อนแนวตั้ง
    zoom_range=0.1,               # สุ่มซูม
    brightness_range=[0.9, 1.1],  # สุ่มปรับระดับความสว่าง
    horizontal_flip=True,         # สุ่มพลิกภาพแนวตั้งหรือแนวนอน
    validation_split=0.2          # กำหนดสัดส่วนแบ่งทำชุดตรวจสอบ 20%
)

val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

# โหลดข้อมูล Train (80%)
train_generator = train_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    seed=42,
    shuffle=True
)

# โหลดข้อมูล Validation (20%)
val_generator = val_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    seed=42,
    shuffle=False
)

num_classes = len(train_generator.class_indices)
print("Detected number of classes:", num_classes)
print("Class Map:", train_generator.class_indices)

## 🧠 ส่วนที่ 7: การสร้าง ฝึกสอน และเปรียบเทียบสถาปัตยกรรมโมเดล Deep Learning
เราทำการเปรียบเทียบโมเดล 3 โมเดล ได้แก่ `MobileNetV2` (น้ำหนักเบา รันบนมือถือดีเยี่ยม), `EfficientNetB0` (ค่าความแม่นยำคุ้มค่าสเกลพารามิเตอร์) และ `ResNet50V2` (โครงสร้างลึกทรงประสิทธิภาพ)

In [ ]:
def build_transfer_learning_model(model_name, input_shape=(224, 224, 3), num_classes=22):
    """
    สร้างแบบจำลองโดยตรึงน้ำหนักโครงสร้างหลักที่ใช้ ImageNet และสวมหัว Classification เพิ่มเติม
    """
    if model_name == "MobileNetV2":
        base_model = tf.keras.applications.MobileNetV2(input_shape=input_shape, include_top=False, weights='imagenet')
    elif model_name == "EfficientNetB0":
        base_model = tf.keras.applications.EfficientNetB0(input_shape=input_shape, include_top=False, weights='imagenet')
    elif model_name == "ResNet50V2":
        base_model = tf.keras.applications.ResNet50V2(input_shape=input_shape, include_top=False, weights='imagenet')
    else:
        raise ValueError("Unknown model name")
        
    # ตรึงโมเดลหลัก (แช่แข็งค่าน้ำหนักฟีเจอร์)
    base_model.trainable = False

    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.5),
        layers.Dense(256, activation='relu'),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    return model, base_model

### 7.1 เฟสแรก: การเรียนรู้สกัดฟีเจอร์ (Feature Extraction Phase)
ในส่วนนี้เราจะล็อคโครงสร้างด้านล่างไว้และเทรนเฉพาะ Classifier ด้านบนเป็นเวลา 20-30 epochs ด้วยอัตราการเรียนรู้เริ่มต้น $10^{-4}$

In [ ]:
histories = {}
trained_models = {}

selected_models = ["MobileNetV2", "EfficientNetB0", "ResNet50V2"]

for name in selected_models:
    print(f"\n=== Start Feature Extraction for {name} ===")
    model, base_model = build_transfer_learning_model(name, num_classes=num_classes)
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=4)
    ]
    
    history = model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=20,
        callbacks=callbacks
    )
    
    histories[name] = history.history
    trained_models[name] = (model, base_model)

### 7.2 เฟสสอง: การปรับแต่งเชิงลึก (Fine-Tuning Phase)
ปลดล็อกโมดูลบล็อกท้ายๆ ของ Base model และฝึกสอนพร้อมกันด้วยอัตราการเรียนรู้ระดับต่ำมาก $10^{-5}$

In [ ]:
fine_tune_histories = {}

for name in selected_models:
    print(f"\n=== Start Fine-Tuning for {name} ===")
    model, base_model = trained_models[name]
    
    base_model.trainable = True
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5)
    ]
    
    history_ft = model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=20,
        callbacks=callbacks
    )
    
    fine_tune_histories[name] = history_ft.history

## 📊 ส่วนที่ 8: พล็อตและเปรียบเทียบผลลัพธ์การฝึกสอน (Evaluation & Visualization)

In [ ]:
plt.figure(figsize=(15, 10))

for idx, name in enumerate(selected_models):
    acc_total = histories[name]['accuracy'] + fine_tune_histories[name]['accuracy']
    val_acc_total = histories[name]['val_accuracy'] + fine_tune_histories[name]['val_accuracy']
    
    plt.subplot(2, 2, idx+1)
    plt.plot(acc_total, label='Train Acc', color='teal', linestyle='-')
    plt.plot(val_acc_total, label='Val Acc', color='coral', linestyle='--')
    plt.axvline(x=len(histories[name]['accuracy']), color='gray', linestyle=':', label='Fine-Tuning Start')
    plt.title(f"{name} Performance Curves")
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

plt.tight_layout()
plt.show()

## 💾 ส่วนที่ 9: บันทึกและแปลงโมเดลที่ดีที่สุดเป็น TensorFlow Lite (.tflite)
โมเดลจำแนกแมลงที่ฝึกสอนสมบูรณ์แล้วจะถูกเซฟเก็บและทำการแปลงเป็น TFLite เพื่อให้สามารถโหลดใช้งานอย่างรวดเร็วและใช้พลังงานต่ำในแอปพลิเคชันพกพาของเกษตรกร

In [ ]:
best_model_name = "EfficientNetB0"
model_to_export, _ = trained_models[best_model_name]

keras_model_path = 'rice_insect_best_model.h5'
model_to_export.save(keras_model_path)
print(f"Saved best model {best_model_name} to {keras_model_path}")

converter = tf.lite.TFLiteConverter.from_keras_model(model_to_export)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

tflite_path = 'rice_insect_classifier.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

print(f"Success: TFLite model generated and saved to {tflite_path}")
print("Size of Keras H5 model:", os.path.getsize(keras_model_path) / (1024*1024), "MB")
print("Size of TFLite model (optimized):", os.path.getsize(tflite_path) / (1024*1024), "MB")

## 🚜 ส่วนที่ 10: ระบบทดสอบการจำแนกและดึงคำแนะนำเชิงปฏิบัติการกำจัดแมลงศัตรูข้าว (Real-time AI Inference & Advice)
จำลองการทำงานบนแอปพลิเคชันพกพาเมื่อผู้ใช้งานถ่ายรูปและอัปโหลดระบบจำแนกจะให้คำแนะนำในการจัดการตามปัญญาประดิษฐ์และคลังภูมิปัญญาของไทยทันที

In [ ]:
def get_advice_for_pest(pest_name_thai, rec_json_path=RECOMMENDATIONS_PATH):
    """
    ฟังก์ชันสืบค้นข้อแนะนำตามคลาสแมลงภาษาไทยจาก recommendations.json
    """
    if not os.path.exists(rec_json_path):
        return "คำแนะนำ: ไม่พบฐานข้อมูลคำแนะนำเกษตรกร กรุณาจัดเตรียมไฟล์ recommendations.json"
        
    with open(rec_json_path, 'r', encoding='utf-8') as f:
        database = json.load(f)
        
    if pest_name_thai in database:
        info = database[pest_name_thai]
        name_2_text = f" / {info['scientific_name_2']}" if info.get('scientific_name_2') else ""
        rec_text = "\n- ".join(info['recommendations'])
        return f"📌 ชื่อภาษาอังกฤษ: {info['english_name']}\n🔬 ชื่อวิทยาศาสตร์: {info['scientific_name']}{name_2_text}\n⚠ รายละเอียด: {info['description']}\n\n✅ คำแนะนำเชิงปฏิบัติ:\n- {rec_text}"
    else:
        return "ไม่พบคู่มือแนะนำเฉพาะเจาะจงสำหรับแมลงชนิดนี้ในฐานข้อมูล"

def predict_and_advise(image_path, model, class_indices, target_size=(224, 224)):
    """
    ทำนายภาพแมลงศัตรูข้าวและแสดงผลคำแนะนำคู่กัน
    """
    processed_img = letterbox_image(image_path, target_size=target_size)
    if processed_img is None:
        print("Error loading image")
        return
        
    img_input = np.expand_dims(processed_img / 255.0, axis=0)
    
    preds = model.predict(img_input)
    pred_idx = np.argmax(preds[0])
    confidence = preds[0][pred_idx] * 100
    
    inverse_map = {v: k for k, v in class_indices.items()}
    pred_class_thai = inverse_map[pred_idx]
    
    plt.figure(figsize=(6, 6))
    plt.imshow(processed_img)
    plt.title(f"Prediction: {pred_class_thai} ({confidence:.2f}%)")
    plt.axis('off')
    plt.show()
    
    print("=================== รายงานและคำแนะนำเชิงปฏิบัติการเกษตร ===================")
    print(f"พบศัตรูข้าวชนิด: {pred_class_thai} (ความมั่นใจ {confidence:.2f}%)")
    advice = get_advice_for_pest(pred_class_thai)
    print(advice)
    print("=======================================================================")

if os.path.exists(DATASET_DIR):
    first_class = classes[0]
    class_dir = os.path.join(DATASET_DIR, first_class)
    test_image = os.path.join(class_dir, os.listdir(class_dir)[0])
    
    model_instance, _ = trained_models["EfficientNetB0"]
    predict_and_advise(test_image, model_instance, train_generator.class_indices)